# Seattle Library Checkouts Analysis

This notebook analyzes the Seattle Library checkouts timestamp stream with the same Poisson-compatible window-selection workflow used for the retail-event dataset.

We parse the timestamp events, aggregate them into hourly counts, and test whether the process is suitable for a non-hurdle Poisson-style window analysis.


In [6]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import csv
import sys

ROOT = Path.cwd().resolve()
candidate_roots = [ROOT, *ROOT.parents]
for candidate in candidate_roots:
    if (candidate / "src" / "count_ts_window_selector").exists():
        ROOT = candidate
        break

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

DATA_FILE = ROOT / "data" / "seattle-library-checkouts_timestamp_events.csv"
print(f"Reading {DATA_FILE}")

timestamps = []
with DATA_FILE.open() as f:
    reader = csv.DictReader(f)
    for row in reader:
        raw = row["timestamp"]
        if "." in raw:
            base, frac = raw.split(".", 1)
            raw = base + "." + frac[:6]
        timestamps.append(datetime.strptime(raw, "%Y-%m-%d %H:%M:%S.%f"))

print(f"Rows: {len(timestamps)}")
print(f"Minimum timestamp: {min(timestamps)}")
print(f"Maximum timestamp: {max(timestamps)}")
print(f"Time span (hours): {(max(timestamps) - min(timestamps)).total_seconds() / 3600:.2f}")


Reading /home/rajiv/programming/count_ts_window_selector/data/seattle-library-checkouts_timestamp_events.csv
Rows: 6226
Minimum timestamp: 2024-03-01 09:01:43.280430
Maximum timestamp: 2024-03-31 17:50:48.467727
Time span (hours): 728.82


In [7]:
hourly_counts = defaultdict(int)
for dt in timestamps:
    key = dt.replace(minute=0, second=0, microsecond=0)
    hourly_counts[key] += 1

series = [hourly_counts[k] for k in sorted(hourly_counts)]
print(f"Hourly bins: {len(series)}")
print(f"Mean hourly count: {sum(series) / len(series):.2f}")
print(f"Min hourly count: {min(series)}")
print(f"Max hourly count: {max(series)}")
print(f"Zero-hour proportion: {sum(v == 0 for v in series) / len(series):.3f}")
print("First 20 hourly counts:", series[:20])


Hourly bins: 279
Mean hourly count: 22.32
Min hourly count: 8
Max hourly count: 33
Zero-hour proportion: 0.000
First 20 hourly counts: [16, 27, 25, 18, 16, 22, 16, 20, 20, 24, 26, 14, 25, 27, 20, 21, 20, 25, 17, 26]


In [8]:
from count_ts_window_selector import (
    WindowSelectionConfig,
    check_data_suitability,
    select_time_window,
)

config = WindowSelectionConfig(
    min_event_rate=5.0,
    max_zero_window_proportion=0.1,
    min_windows=30,
    confidence_level=0.90,
)

print("Suitability check:", check_data_suitability(series, min_event_rate=config.min_event_rate, max_zero_window_proportion=config.max_zero_window_proportion))

candidate_windows = [1, 2, 3, 4, 6, 8, 12, 24]
results = select_time_window(series, candidate_windows, config=config)
print("Selected Poisson-compatible windows:")
for result in results:
    print(result)


Suitability check: {'observed_event_rate': 22.315412186379927, 'zero_window_proportion': 0.0}
Selected Poisson-compatible windows:
{'window_size': 1, 'fano_factor': 0.9485078817768144, 'fano_lower': 0.8646934442215072, 'fano_upper': 1.1434834289825568}
{'window_size': 2, 'fano_factor': 0.8473896707041003, 'fano_lower': 0.810557727544271, 'fano_upper': 1.2059079633990153}
{'window_size': 3, 'fano_factor': 0.8963602843615134, 'fano_lower': 0.7704518623681242, 'fano_upper': 1.2542368446550745}
{'window_size': 4, 'fano_factor': 0.7557803743662079, 'fano_lower': 0.7355916655042538, 'fano_upper': 1.2977965356157959}
{'window_size': 6, 'fano_factor': 0.800720097933319, 'fano_lower': 0.6802724254576773, 'fano_upper': 1.370138519472879}
{'window_size': 8, 'fano_factor': 0.6349349079319405, 'fano_lower': 0.6323192118398421, 'fano_upper': 1.4363601187600281}


## Interpretation

The Seattle library checkout stream is dense enough for the non-hurdle workflow, with a mean hourly rate of about 22.3 events/hour and no zero-hour windows. The Fano factors remain close to 1 across the compatible windows, and the 1-hour window is the closest match to the Poisson reference of 1.

This indicates the checkouts are reasonably well described as a near-Poisson process over a 1- to 6-hour observation horizon, with the 1-hour aggregate providing the most defensible default window choice.


## Visual diagnostics

The next cells plot the hourly checkout series and summarize how the Fano factor changes across candidate aggregate windows.


In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hour_labels = sorted(hourly_counts)
hour_values = [hourly_counts[k] for k in hour_labels]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(hour_labels, hour_values, color="#2f70af", linewidth=1.5)
ax.set_title("Hourly Seattle library checkouts")
ax.set_xlabel("Hour bucket")
ax.set_ylabel("Checkouts")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


/tmp/ipykernel_142535/2049317074.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
import numpy as np
from count_ts_window_selector import WindowSelectionConfig, select_time_window

config = WindowSelectionConfig(
    min_event_rate=5.0,
    max_zero_window_proportion=0.1,
    min_windows=30,
    confidence_level=0.90,
)
results = select_time_window(series, [1, 2, 3, 4, 6, 8, 12, 24], config=config)
windows = [r["window_size"] for r in results]
fanos = [r["fano_factor"] for r in results]
lower = [r["fano_lower"] for r in results]
upper = [r["fano_upper"] for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
yerr = [np.array(fanos) - np.array(lower), np.array(upper) - np.array(fanos)]
ax.errorbar(windows, fanos, yerr=yerr, fmt="o-", color="#2f70af", capsize=4)
ax.axhline(1.0, linestyle="--", color="#d65f5f", linewidth=1.2, label="Poisson reference (Fano = 1)")
ax.set_title("Fano factor vs. aggregation window")
ax.set_xlabel("Window size (hours)")
ax.set_ylabel("Fano factor")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


/tmp/ipykernel_142535/2964166938.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
